# Week 5 Final Submission - AI-Assisted Triage Data Exploration

**Purpose:** Explore the ED triage dataset, document data quality, produce a dashboard, and create evidence for the 3-page feasibility memo.

**Target:** `esi` (Emergency Severity Index), where **1 = most urgent** and **5 = least urgent**.

**Important safety rule:** Never paste raw patient rows into AI tools. Use only summary statistics, schemas and aggregate tables.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

# Change this if your CSV is in a different folder.
DATA_PATH = Path("../data/yaleemmlc_admissionprediction_triage.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("yaleemmlc_admissionprediction_triage.csv")

FIG_DIR = Path("../docs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Ready")

## 1. Load the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} patient records x {df.shape[1]:,} columns")
df.head()

## 2. Schema map and feature families

In [ ]:
TARGET = "esi"

VITALS = [
    "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
    "triage_vital_o2", "triage_vital_o2_device", "triage_vital_temp", "triage_glucose"
]

DEMOGRAPHICS = ["age", "gender", "ethnicity", "race", "lang", "religion", "maritalstatus", "employstatus", "insurance_status"]
ADMIN = ["dep_name", "arrivalmode", "arrivalmonth", "arrivalday", "arrivalhour_bin"]
LEAKAGE = ["disposition", "previousdispo"]  # known after triage; do not use as model inputs
CC_COLS = [c for c in df.columns if c.startswith("cc_")]

print("Target:", TARGET)
print("Vital columns:", len(VITALS))
print("Chief complaint flags:", len(CC_COLS))
print("Leakage columns to exclude:", LEAKAGE)

## 3. Clean working copy

In [ ]:
clean = df.copy()

# Remove export/index artifact if present.
if "Unnamed: 0" in clean.columns:
    clean = clean.drop(columns=["Unnamed: 0"])

# ESI is a triage class, so store it as an integer.
clean[TARGET] = clean[TARGET].round().astype(int)

clean.to_csv("../data_cleaned_week5.csv", index=False)
print(f"Cleaned working file saved with {clean.shape[0]:,} rows x {clean.shape[1]:,} columns")

## 4. Missingness and duplicates

In [ ]:
missing = (
    df.isna()
      .mean()
      .mul(100)
      .round(2)
      .sort_values(ascending=False)
      .rename("missing_pct")
)

missing_table = missing.reset_index().rename(columns={"index": "column"})
missing_table["missing_count"] = df.isna().sum().values

print("Total missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicate rows excluding original index:", int(df.drop(columns=["Unnamed: 0"], errors="ignore").duplicated().sum()))
missing_table.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
top_missing = missing_table.head(15)
ax.barh(top_missing["column"][::-1], top_missing["missing_pct"][::-1])
ax.set_title("Top 15 Columns by Missingness")
ax.set_xlabel("Missing values (%)")
ax.text(0.5, 0.05, "No missing values detected across the supplied file.",
        transform=ax.transAxes, ha="center", va="bottom",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_1_missingness.png", dpi=150)
plt.show()

## 5. Target distribution

In [ ]:
esi_counts = clean[TARGET].value_counts().sort_index()
esi_pct = (esi_counts / len(clean) * 100).round(2)
pd.DataFrame({"count": esi_counts, "percent": esi_pct})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(esi_counts.index.astype(str), esi_counts.values)
ax.set_title("ESI Triage Level Distribution")
ax.set_xlabel("ESI level (1 = most urgent, 5 = least urgent)")
ax.set_ylabel("Patient count")
for i, (level, count) in enumerate(esi_counts.items()):
    ax.text(i, count + 500, f"{count:,}\n({count/len(clean)*100:.1f}%)", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_2_esi_distribution.png", dpi=150)
plt.show()

## 6. Demographics and fairness lens

In [ ]:
for col in ["age", "gender", "race", "ethnicity", "lang", "insurance_status"]:
    print("\n", col.upper())
    if pd.api.types.is_numeric_dtype(clean[col]):
        print(clean[col].describe().round(1))
    else:
        print(clean[col].value_counts(dropna=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

race_counts = clean["race"].value_counts()
eth_counts = clean["ethnicity"].value_counts()

axes[0].barh(race_counts.index[::-1], race_counts.values[::-1])
axes[0].set_title("Race Distribution")
axes[0].set_xlabel("Patient count")

axes[1].barh(eth_counts.index[::-1], eth_counts.values[::-1])
axes[1].set_title("Ethnicity Distribution")
axes[1].set_xlabel("Patient count")

plt.tight_layout()
plt.savefig(FIG_DIR / "figure_4_race_ethnicity.png", dpi=150)
plt.show()

## 7. Vital signs and outliers

In [ ]:
clean[VITALS].describe().T.round(1)

In [ ]:
# Plausible wide bounds for audit, not automatic deletion.
PLAUSIBLE = {
    "triage_vital_hr": (30, 220),
    "triage_vital_sbp": (50, 260),
    "triage_vital_dbp": (30, 160),
    "triage_vital_rr": (5, 60),
    "triage_vital_o2": (50, 100),
    "triage_vital_temp": (90, 106),
    "triage_glucose": (20, 1000),
}

outlier_rows = []
for col, (low, high) in PLAUSIBLE.items():
    mask = (clean[col] < low) | (clean[col] > high)
    outlier_rows.append({
        "feature": col,
        "low_bound": low,
        "high_bound": high,
        "outside_count": int(mask.sum()),
        "outside_percent": round(mask.mean() * 100, 3),
        "min": clean[col].min(),
        "max": clean[col].max()
    })

outlier_table = pd.DataFrame(outlier_rows)
outlier_table

In [ ]:
vital_means = clean.groupby(TARGET)[["triage_vital_hr", "triage_vital_rr", "triage_vital_o2", "triage_vital_temp", "triage_glucose"]].mean()

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), vital_means.columns):
    ax.plot(vital_means.index, vital_means[col], marker="o")
    ax.set_title(col)
    ax.set_xlabel("ESI level")
    ax.set_ylabel("Mean value")

axes.ravel()[-1].axis("off")
plt.suptitle("Mean Vital Signs by ESI Level")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_7_vitals_by_esi.png", dpi=150)
plt.show()

## 8. Chief complaints

In [ ]:
cc_sum = clean[CC_COLS].sum().sort_values(ascending=False)
top_cc = pd.DataFrame({
    "complaint": cc_sum.index,
    "count": cc_sum.values,
    "prevalence_pct": (cc_sum.values / len(clean) * 100).round(2)
})
top_cc.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top10 = cc_sum.head(10)
labels = [c.replace("cc_", "").replace("-", " ").replace(">", " > ") for c in top10.index]
ax.barh(labels[::-1], top10.values[::-1])
ax.set_title("Top 10 Chief Complaint Flags")
ax.set_xlabel("Patient count")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_5_top_complaints.png", dpi=150)
plt.show()

## 9. Correlation and feature shortlist

In [ ]:
# For ESI, lower values mean more urgent. Negative correlation means the feature is associated with higher acuity.
num_features = ["age"] + VITALS
num_corr = pd.Series({c: clean[c].corr(clean[TARGET], method="spearman") for c in num_features})

cc_corr = pd.Series({
    c: clean[c].corr(clean[TARGET]) if clean[c].std() > 0 else np.nan
    for c in CC_COLS
}).dropna()

print("Vitals/demographics most associated with ESI:")
display(num_corr.sort_values(key=lambda s: s.abs(), ascending=False).round(3))

print("Chief complaints most associated with ESI:")
display(cc_corr.sort_values(key=lambda s: s.abs(), ascending=False).head(15).round(3))

In [ ]:
feature_shortlist = pd.DataFrame([
    [1, "triage_vital_o2_device", -0.219, "Need for supplemental oxygen is a strong signal of respiratory compromise."],
    [2, "triage_vital_o2", 0.159, "Low oxygen saturation directly signals hypoxia."],
    [3, "age", -0.238, "Older adults have higher risk of deterioration and admission."],
    [4, "cc_chestpain", -0.164, "May represent acute coronary syndrome or another high-risk cause."],
    [5, "cc_shortnessofbreath", -0.150, "May represent respiratory failure, heart failure, asthma/COPD or sepsis."],
    [6, "triage_vital_hr", -0.073, "Tachycardia or bradycardia can signal shock, fever, pain or arrhythmia."],
    [7, "triage_glucose", -0.111, "Abnormal glucose can signal diabetic or metabolic emergency."],
    [8, "cc_alteredmentalstatus", -0.132, "High-risk neurologic, septic, toxicologic or metabolic presentation."],
    [9, "triage_vital_rr", -0.056, "Early marker of respiratory or metabolic deterioration."],
    [10, "cc_suicidal", -0.143, "Safety-critical mental health presentation requiring urgent assessment."]
], columns=["rank", "feature", "simple_association_with_esi", "clinical_reasoning"])

feature_shortlist

## 10. Final dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

top_missing = missing_table.head(15)
axes[0,0].barh(top_missing["column"][::-1], top_missing["missing_pct"][::-1])
axes[0,0].set_title("Missingness (top 15)")
axes[0,0].set_xlabel("Missing %")
axes[0,0].text(0.5, 0.1, "0% missing in all columns", transform=axes[0,0].transAxes, ha="center",
               bbox=dict(facecolor="white", alpha=.8))

axes[0,1].bar(esi_counts.index.astype(str), esi_counts.values)
axes[0,1].set_title("ESI Distribution")
axes[0,1].set_xlabel("ESI")
axes[0,1].set_ylabel("Count")

axes[1,0].barh(labels[::-1], top10.values[::-1])
axes[1,0].set_title("Top 10 Chief Complaints")
axes[1,0].set_xlabel("Count")

axes[1,1].hist(clean["age"], bins=30, edgecolor="black")
axes[1,1].set_title("Age Distribution")
axes[1,1].set_xlabel("Age")
axes[1,1].set_ylabel("Count")

plt.suptitle("Week 5 Data Quality Dashboard")
plt.tight_layout()
plt.savefig(FIG_DIR / "dashboard_4_plots.png", dpi=150)
plt.show()

## 11. Final conclusion

**Verdict:** Proceed with caveats.

The dataset is strong enough for Week 6 baseline modelling because it is large, has complete ESI labels, and includes clinically meaningful triage variables. However, it is not ready for clinical deployment. The next model must remove leakage variables, handle class imbalance, audit extreme values, and test subgroup performance before any real-world use.
